# FUMA input preparation

FUMA wants a small, strictly-formatted table: `rsID A1 A2 BETA SE N P`, one
row per variant, no missing values. This converts a LocusZoom-format export
(from `locuszoom_prep.ipynb`) into that, applying the frequency and rsID
filters FUMA needs.

Notes on the filters:

* FUMA needs a real rsID, so variants that are only positional are dropped.
* MAF should be above 0.01 but 0.05 is what we used below
* `N` is not in the LocusZoom export, so the per-analysis sample size has to be
  supplied here - take it from the covariate file the GWAS actually ran on.

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# The shared pipeline config lives in the repo root; point GWAS_SCRIPTS_DIR
# at it if this notebook is run from somewhere else.
SCRIPTS_DIR = Path(os.environ.get('GWAS_SCRIPTS_DIR', Path.cwd().parent))
sys.path.insert(0, str(SCRIPTS_DIR))

import gwas_config as cfg

pd.options.mode.chained_assignment = None
cfg.describe()

## Load the summary statistics

In [ ]:
LABEL = 'EUR'
ANALYSIS = 'SURV'

# LocusZoom-format export written by locuszoom_prep.ipynb.
IN_PATH = Path(f'locuszoom/R{cfg.RELEASE}/{LABEL}_{ANALYSIS}_R{cfg.RELEASE}.gz')

# Sample size of the GWAS that produced these statistics
N_SAMPLES = None

MAF_MIN = 0.05   # FUMA needs > 0.01; 0.05 is the usual working threshold

df = pd.read_csv(IN_PATH, compression='gzip', sep='\t', engine='c')
print(f'{len(df):,} variants')
display(df.head())

In [ ]:
# Locuszoom export carries -log10(P) so convert to p-value
df['P'] = 10 ** (-df['neg_log_pvalue'])
display(df.alt_allele_freq.describe())

## Filter

`alt_allele_freq` is the ALT allele frequency, with the effect allele == ALT in our Locuszoom files

In [ ]:
fuma = df[df.rsid.str.startswith('rs', na=False)]
print(f'{len(fuma):,} with an rsID')

fuma = fuma[['rsid', 'ref', 'alt', 'alt_allele_freq', 'beta', 'stderr_beta', 'P']].dropna()
print(f'{len(fuma):,} with complete statistics')

maf = np.minimum(fuma.alt_allele_freq, 1 - fuma.alt_allele_freq)
fuma = fuma[maf > MAF_MIN]
print(f'{len(fuma):,} with MAF > {MAF_MIN}')

In [ ]:
if N_SAMPLES is None:
    raise ValueError('set N_SAMPLES to the sample size of this GWAS')

fuma = fuma.rename(columns={'rsid': 'rsID', 'alt': 'A1', 'ref': 'A2',
                            'beta': 'BETA', 'stderr_beta': 'SE'})
fuma['N'] = N_SAMPLES

fuma = fuma[['rsID', 'A1', 'A2', 'BETA', 'SE', 'N', 'P']]
display(fuma.head())
print(fuma.shape)

In [ ]:
out_dir = Path('fuma_inputs')
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / f'{LABEL}_{ANALYSIS}_R{cfg.RELEASE}.tsv.gz'
fuma.to_csv(out_path, compression='gzip', sep='\t', index=False)
print(f'wrote {out_path}')